# ML-Enabled Relaxed Garment Ranking System

## Problem Statement

Traditional e-commerce systems rely on strict filtering:
If a product does not exactly match selected filters, it is removed.

This rigid approach hides near-relevant products and limits discovery.

To solve this, I designed a Relaxed Filtering System:
- Convert filters into similarity features
- Rank instead of eliminate
- Show exact matches first
- Keep near-relevant products visible

Initially, ranking used manually assigned weights.
However, manual weights are subjective.

This project upgrades the system into a:
Data-driven, ML-based probabilistic ranking engine.

The model learns feature importance automatically from interaction behavior.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

## Step 1: Load Garment Product Catalog

We use a pre-generated structured dataset of 50,000 garment products.
The dataset contains realistic category, price, fabric, size, and rating distributions.

In [2]:
file_path = "/kaggle/input/datasets/srishanthkargaom/garments-50k-dataset-csv/garments_50k_dataset.csv"


df = pd.read_csv(file_path)


if "unnamed: 1" in df.columns.str.lower().tolist() or len(df.columns) <= 2:
    df = pd.read_csv(file_path, header=1)

# Cleaning column names
df.columns = df.columns.str.strip().str.lower()

# Remove fully empty columns
df = df.loc[:, ~df.columns.str.contains("^unnamed")]

print("Final Columns:", df.columns.tolist())
print("Dataset Shape:", df.shape)

df.head()

Final Columns: ['product_id', 'gender', 'category', 'brand', 'color', 'fabric', 'size', 'price', 'discount_percent', 'rating', 'stock']
Dataset Shape: (50000, 11)


,product_id,gender,category,brand,color,fabric,size,price,discount_percent,rating,stock
0,1,Women,T-Shirt,Mango,Pink,Cotton,S,26.23,12,3.7,30
1,2,Women,Kurta,Topshop,Navy,Polyester,M,41.17,32,3.6,337
2,3,Men,Trousers,Old Navy,Yellow,Linen,M,59.92,41,3.9,173
3,4,Women,Jeans,Topshop,Beige,Denim,L,63.74,42,4.8,181
4,5,Men,Shirt,Old Navy,Brown,Polyester,S,36.48,26,4.1,128


## Step 2: Traditional Manual Weighted Ranking

Earlier, ranking was done using manually defined weights:

Score =
0.4 × category_match +
0.3 × color_match +
0.2 × fabric_match +
0.1 × price_similarity

Limitations:
- Subjective weight selection
- No learning from user behavior
- Not adaptive

## Step 3: Simulate User Filters

We simulate 1,000 users applying filters:
- Preferred category
- Preferred color
- Preferred fabric
- Preferred size
- Budget

In [3]:
np.random.seed(42)

n_users = 1000

users = pd.DataFrame({
    "user_id": range(1,n_users+1),
    "preferred_category": np.random.choice(df["category"].unique(),n_users),
    "preferred_color": np.random.choice(df["color"].unique(),n_users),
    "preferred_fabric": np.random.choice(df["fabric"].unique(),n_users),
    "preferred_size": np.random.choice(df["size"].unique(),n_users),
    "budget": np.random.randint(500,5000,n_users)
})

users.head()

,user_id,preferred_category,preferred_color,preferred_fabric,preferred_size,budget
0,1,Jeans,White,Wool,XL,3000
1,2,Shirt,Brown,Wool,M,4145
2,3,Trousers,White,Linen,XL,3145
3,4,Shirt,White,Denim,M,2849
4,5,Shirt,Red,Polyester,L,2432


## Step 4: Convert Filters into Similarity Features

Instead of strict filtering, we compute:

- category_match
- color_match
- fabric_match
- size_match
- price_similarity
- rating_normalized

These features represent relaxed filtering logic.

In [4]:
samples_per_user = 200
interaction_data = []

for _, user in users.iterrows():
    sampled = df.sample(samples_per_user)

    for _, product in sampled.iterrows():
        category_match = int(user["preferred_category"] == product["category"])
        color_match = int(user["preferred_color"] == product["color"])
        fabric_match = int(user["preferred_fabric"] == product["fabric"])
        size_match = int(user["preferred_size"] == product["size"])

        price_similarity = 1 - abs(product["price"] - user["budget"]) / user["budget"]
        price_similarity = max(0, price_similarity)

        interaction_data.append([
            category_match,
            color_match,
            fabric_match,
            size_match,
            price_similarity,
            product["rating"]/5
        ])

interaction_df = pd.DataFrame(interaction_data, columns=[
    "category_match","color_match","fabric_match",
    "size_match","price_similarity","rating_normalized"
])

## Step 5: Simulate Purchase Outcomes

Users are more likely to purchase products that:
- Match category
- Match color
- Align with budget
- Have higher rating

We simulate purchase probability using a logistic function.

In [5]:
logits = (
    1.2*interaction_df["category_match"] +
    1.0*interaction_df["color_match"] +
    0.8*interaction_df["fabric_match"] +
    0.8*interaction_df["price_similarity"] +
    0.3*interaction_df["rating_normalized"]
    - 2.5   
)

prob = 1/(1+np.exp(-logits))
interaction_df["clicked"] = np.random.binomial(1,prob)

## Step 6: Learn Feature Importance Automatically

Logistic Regression replaces manual weights.
It learns coefficients by minimizing cross-entropy loss.

In [6]:
features = interaction_df.columns[:-1]

X = interaction_df[features]
y = interaction_df["clicked"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test,y_pred))
print("ROC-AUC:",roc_auc_score(y_test,y_prob))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92     33823
           1       0.60      0.01      0.02      6177

    accuracy                           0.85     40000
   macro avg       0.72      0.51      0.47     40000
weighted avg       0.81      0.85      0.78     40000

ROC-AUC: 0.6644304731249284


In [7]:
coef_df = pd.DataFrame({
    "Feature":features,
    "Coefficient":model.coef_[0]
}).sort_values(by="Coefficient",ascending=False)

coef_df

,Feature,Coefficient
0,category_match,1.199609
1,color_match,0.998568
2,fabric_match,0.820668
5,rating_normalized,0.342876
4,price_similarity,0.190980
3,size_match,0.014297


## Step 7: Relaxed Ranking (Exact + Near-Relevant)

We do NOT strictly filter products.

Instead:
- Compute similarity features for ALL products
- Predict purchase probability
- Rank by probability

Exact matches appear first.
Near-relevant products follow.

In [8]:
random_user = users.sample(1).iloc[0]

candidate = df.copy()

candidate["category_match"] = (candidate["category"]==random_user["preferred_category"]).astype(int)
candidate["color_match"] = (candidate["color"]==random_user["preferred_color"]).astype(int)
candidate["fabric_match"] = (candidate["fabric"]==random_user["preferred_fabric"]).astype(int)
candidate["size_match"] = (candidate["size"]==random_user["preferred_size"]).astype(int)

candidate["price_similarity"] = 1 - abs(candidate["price"]-random_user["budget"])/random_user["budget"]
candidate["price_similarity"] = candidate["price_similarity"].clip(0,1)

candidate["rating_normalized"] = candidate["rating"]/5

candidate["click_probability"] = model.predict_proba(
    candidate[features]
)[:,1]

top_20 = candidate.sort_values(
    by="click_probability",ascending=False
).head(20)

top_20[["product_id","category","color","price","click_probability"]]

,product_id,category,color,price,click_probability
24867,24868,Kurta,Green,73.78,0.699057
16128,16129,Kurta,Green,44.77,0.696047
20495,20496,Kurta,Green,47.31,0.695945
10010,10011,Kurta,Green,51.67,0.694652
9472,9473,Kurta,Green,39.19,0.694549
4608,4609,Kurta,Green,44.31,0.694467
15773,15774,Kurta,Green,73.21,0.693250
47821,47822,Kurta,Green,69.82,0.693222
11369,11370,Kurta,Green,64.68,0.693179
1102,1103,Kurta,Green,48.32,0.693167


## Conclusion

This system demonstrates:

- Transition from manual weighted ranking to ML-based ranking
- Automatic learning of feature importance
- Continuous relevance scoring
- Exact matches ranked highest
- Near-relevant products remain visible

Training Objective:
Binary classification

Deployment Objective:
Probabilistic ranking

Relaxed filtering remains intact.
Manual subjectivity is eliminated.
Machine learning learns relevance.